In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report

In [ ]:
MODEL_NAME = "monologg/bert-base-cased-goemotions-original"

tokenizer = AutoTokenizer.from_pretrained("goemotions_model", local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained("goemotions_model", local_files_only=True)
model.eval()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# 1. Завантажуємо GoEmotions
dataset = load_dataset("go_emotions", "simplified")
test_ds = dataset["test"]
print(test_ds)

In [ ]:
label2id = model.config.label2id
id2label = model.config.id2label
num_labels = len(label2id)

In [ ]:
def preprocess(batch):
    # Токенізація
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # перетворюємо список id-міток у multi-hot вектор
    labels_multi = np.zeros((len(batch["labels"]), num_labels), dtype=int)
    for i, lbls in enumerate(batch["labels"]):
        for lid in lbls:
            labels_multi[i, lid] = 1
    enc["labels"] = labels_multi.tolist()
    return enc

In [ ]:
test_enc = test_ds.map(preprocess, batched=True)

In [ ]:
# Налаштовуємо формат для PyTorch
test_enc.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32
test_loader = DataLoader(test_enc, batch_size=batch_size)

In [27]:
from tqdm import tqdm

all_preds = []
all_labels = []


# Додаємо прогрес-бар
pbar = tqdm(test_loader, desc="Evaluating", unit="batch")

# 4. Прогін через модель
with torch.no_grad():
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].numpy()  # справжні мітки

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu().numpy()

        # multi-label: sigmoid + поріг
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)
        
         # необов’язково – можна показувати скільки оброблено
        pbar.set_postfix({
            "processed": len(all_preds),
        })
        
y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

Evaluating: 100%|██████████| 170/170 [44:15<00:00, 15.62s/batch, processed=170]


In [28]:
# 5. Метрики по кожній емоції
target_names = [id2label[i] for i in range(num_labels)]

report = classification_report(
    y_true,
    y_pred,
    target_names=target_names,
    zero_division=0,
    output_dict=True
)

df = pd.DataFrame(report).T[["precision", "recall", "f1-score"]]

# 6. macro-average (беремо рядок 'macro avg' і перейменовуємо)
macro_row = df.loc["macro avg"].rename("macro-average")

# 7. std по всіх емоціях (без службових рядків)
emotion_rows = df.iloc[:num_labels]  # перші num_labels рядків – емоції
std_row = emotion_rows.std().rename("std")

df_final = pd.concat([emotion_rows, macro_row.to_frame().T, std_row.to_frame().T])

print(df_final)

                precision    recall  f1-score
admiration       0.645161  0.714286  0.677966
amusement        0.743506  0.867424  0.800699
anger            0.537975  0.429293  0.477528
annoyance        0.356164  0.325000  0.339869
approval         0.360000  0.384615  0.371901
caring           0.429752  0.385185  0.406250
confusion        0.445205  0.424837  0.434783
curiosity        0.487342  0.542254  0.513333
desire           0.568966  0.397590  0.468085
disappointment   0.284553  0.231788  0.255474
disapproval      0.384328  0.385768  0.385047
disgust          0.592233  0.495935  0.539823
embarrassment    0.571429  0.432432  0.492308
excitement       0.404040  0.388350  0.396040
fear             0.637363  0.743590  0.686391
gratitude        0.915698  0.894886  0.905172
grief            0.000000  0.000000  0.000000
joy              0.640288  0.552795  0.593333
love             0.747082  0.806723  0.775758
nervousness      0.375000  0.260870  0.307692
optimism         0.571429  0.53763